In [ ]:
# Step 0: Colab setup & Drive mount (optional)
# If running in Google Colab, mount Drive so outputs persist.
# If running locally, set DRIVE_MODE = False and use local paths.

DRIVE_MODE = True         # set False if you don't want Drive
DRIVE_DIR = "/content/drive/MyDrive/malware_project"  # change as needed
LOCAL_OUTPUT = "/content/malware_embedding_output"

if DRIVE_MODE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_DIR
else:
    OUTPUT_DIR = LOCAL_OUTPUT

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)


Mounted at /content/drive
Outputs will be saved to: /content/drive/MyDrive/malware_project


In [ ]:
# Step 1: Upload your dataset CSV file
from google.colab import files
uploaded = files.upload()

import pandas as pd

# Load the first uploaded CSV file
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Uploaded file:", filename)
df.head()


Saving feature_vectors_syscallsbinders_frequency_5_Cat.csv to feature_vectors_syscallsbinders_frequency_5_Cat.csv
Uploaded file: feature_vectors_syscallsbinders_frequency_5_Cat.csv


,ACCESS_PERSONAL_INFO___,ALTER_PHONE_STATE___,ANTI_DEBUG_____,CREATE_FOLDER_____,CREATE_PROCESS`_____,CREATE_THREAD_____,DEVICE_ACCESS_____,EXECUTE_____,FS_ACCESS____,FS_ACCESS()____,...,utimes,vfork,vibrate,vibratePattern,wait4,watchRotation,windowGainedFocus,write,writev,Class
0,1,0,0,3,0,14,2,0,3,0,...,0,0,0,0,0,0,0,37,10,1
1,3,0,0,6,0,42,91,0,32,0,...,0,0,0,0,0,0,2,2838,46,1
2,2,0,0,4,0,23,3,0,17,2,...,0,0,0,0,0,0,1,111,20,1
3,1,0,0,4,0,27,9,0,36,0,...,0,0,0,0,0,0,7,987,197,1
4,3,0,0,11,0,18,3,0,16,0,...,0,0,0,0,0,0,1,98,25,1


In [ ]:
df = pd.read_csv("/content/feature_vectors_syscallsbinders_frequency_5_Cat.csv")

# Convert labels: 1–5 → 0–4
df["Class"] = df["Class"] - 1

# Split features and labels
X = df.drop("Class", axis=1)
y = df["Class"]


In [ ]:
# ---------- Step 2: Preprocess + Select 400 Features + Split ----------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------
# 1️⃣ Separate features and labels
# -------------------------------------------------------
X = df.drop("Class", axis=1)
y = df["Class"]

print("Original feature count:", X.shape[1])

# -------------------------------------------------------
# 2️⃣ Compute correlation matrix
# -------------------------------------------------------
corr_matrix = X.corr().abs()

# -------------------------------------------------------
# 3️⃣ Select only the upper triangle (to avoid duplicate comparisons)
# -------------------------------------------------------
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# -------------------------------------------------------
# 4️⃣ Identify columns to drop (correlation > 0.90)
# -------------------------------------------------------
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

print("Highly correlated features detected:", len(to_drop))

# Drop only correlated features
X_reduced = X.drop(columns=to_drop)
print("Remaining after correlation filter:", X_reduced.shape[1])

# -------------------------------------------------------
# 5️⃣ Now select EXACTLY 400 features
#    (If reduced > 400 → pick first 400)
#    (If reduced < 400 → pick all, CNN will use that number)
# -------------------------------------------------------
desired_feature_count = 400

current_feature_count = X_reduced.shape[1]

if current_feature_count > desired_feature_count:
    # Select first 400 columns
    selected_features = X_reduced.iloc[:, :desired_feature_count]
    print(f"Selected EXACTLY {desired_feature_count} features.")
else:
    selected_features = X_reduced
    print(f"Only {current_feature_count} available — using all.")

X = selected_features

print("Final feature count for model:", X.shape[1])

# -------------------------------------------------------
# 6️⃣ Split dataset 60/20/20
# -------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

# -------------------------------------------------------
# 7️⃣ Scale features (fit only on train)
# -------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")


Original feature count: 470
Highly correlated features detected: 96
Remaining after correlation filter: 374
Only 374 available — using all.
Final feature count for model: 374
Train size: (6958, 374)
Validation size: (2320, 374)
Test size: (2320, 374)
Scaling completed.


In [ ]:
# ---- Step: Save ONLY RAW (UNSCALED) TRAIN / VAL / TEST SPLITS ----

import os
import pandas as pd

# Ensure directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Convert raw splits to DataFrames
df_train_raw = pd.DataFrame(X_train)
df_train_raw["Class"] = y_train.values

df_val_raw = pd.DataFrame(X_val)
df_val_raw["Class"] = y_val.values

df_test_raw = pd.DataFrame(X_test)
df_test_raw["Class"] = y_test.values

# Save CSVs
train_split_path = os.path.join(OUTPUT_DIR, "train_374Features.csv")
val_split_path   = os.path.join(OUTPUT_DIR, "val_374Features.csv")
test_split_path  = os.path.join(OUTPUT_DIR, "test_374Features.csv")

df_train_raw.to_csv(train_split_path, index=False)
df_val_raw.to_csv(val_split_path, index=False)
df_test_raw.to_csv(test_split_path, index=False)

print("Saved RAW split CSVs:")
print(train_split_path)
print(val_split_path)
print(test_split_path)


Saved RAW split CSVs:
/content/drive/MyDrive/malware_project/train_374Features.csv
/content/drive/MyDrive/malware_project/val_374Features.csv
/content/drive/MyDrive/malware_project/test_374Features.csv


In [ ]:
# -------- Step 3: Reshape + Build 1D-CNN Model --------

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# Reshape for CNN (add channel=1)
X_train_cnn = X_train_scaled.reshape(-1, X_train_scaled.shape[1], 1)
X_val_cnn   = X_val_scaled.reshape(-1, X_val_scaled.shape[1], 1)
X_test_cnn  = X_test_scaled.reshape(-1, X_test_scaled.shape[1], 1)

print("CNN Input Shape:", X_train_cnn.shape)

# ------- Build 1D CNN Model -------
model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_scaled.shape[1], 1)),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation='relu', name="embedding_layer"),   # <-- This produces your embeddings
    Dropout(0.3),

    Dense(5, activation='softmax')    # 5 classes in CICMalDroid-2020
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ------- Train the model -------
history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=20,
    batch_size=32
)


CNN Input Shape: (6958, 374, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 372, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 186, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 184, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 92, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 11776)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 128)            │     1,507,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,533,061 (5.85 MB)

 Trainable params: 1,533,061 (5.85 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 18s 76ms/step - accuracy: 0.6495 - loss: 1.0341 - val_accuracy: 0.7780 - val_loss: 1.4778
Epoch 2/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 16s 71ms/step - accuracy: 0.8101 - loss: 0.5924 - val_accuracy: 0.8284 - val_loss: 0.6437
Epoch 3/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - accuracy: 0.8597 - loss: 0.4376 - val_accuracy: 0.8586 - val_loss: 1.9034
Epoch 4/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 14s 65ms/step - accuracy: 0.8836 - loss: 0.3647 - val_accuracy: 0.8603 - val_loss: 8.1247
Epoch 5/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 14s 65ms/step - accuracy: 0.8994 - loss: 0.3073 - val_accuracy: 0.8759 - val_loss: 8.7383
Epoch 6/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 16s 72ms/step - accuracy: 0.9036 - loss: 0.2901 - val_accuracy: 0.8892 - val_loss: 7.8706
Epoch 7/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 15s 69ms/step - accuracy: 0.9151 - loss: 0.2617 - val_accuracy: 0.8884 - val_loss: 18.5089
Epoch 8/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 20s 69ms/step - accuracy: 0.9238 - loss: 0.2415 -

In [ ]:
# -------- Step 4: Create Embedding Model and Generate Embeddings (400 Features) --------

from tensorflow.keras.models import Model
import numpy as np
import pandas as pd
import tensorflow as tf # Ensure tf is imported

# Get the input shape from trained model
input_shape = (X_train_cnn.shape[1], 1)

# Define a new Input layer that matches the original model's input
input_tensor = tf.keras.Input(shape=input_shape)

# Create a variable to hold the output of the current layer
x = input_tensor

# Iterate through the layers of the *trained* sequential model
# and apply each layer to `x` until the embedding layer is reached.
for layer in model.layers:
    x = layer(x)
    if layer.name == "embedding_layer":
        break # Stop after the embedding layer

# Create the embedding model
embedding_model = Model(inputs=input_tensor, outputs=x)

# --- Generate embeddings ---
train_embeddings = embedding_model.predict(X_train_cnn, batch_size=32)
val_embeddings   = embedding_model.predict(X_val_cnn, batch_size=32)
test_embeddings  = embedding_model.predict(X_test_cnn, batch_size=32)

print("Train embedding shape:", train_embeddings.shape)
print("Validation embedding shape:", val_embeddings.shape)
print("Test embedding shape:", test_embeddings.shape)

# --- Save embeddings ---
pd.DataFrame(train_embeddings).to_csv("train_embeddings_374Features.csv", index=False)
pd.DataFrame(val_embeddings).to_csv("val_embeddings_374Features.csv", index=False)
pd.DataFrame(test_embeddings).to_csv("test_embeddings_374Features.csv", index=False)

print("Embedding extraction completed and files saved.")

218/218 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step
73/73 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step
Train embedding shape: (6958, 128)
Validation embedding shape: (2320, 128)
Test embedding shape: (2320, 128)
Embedding extraction completed and files saved.


In [ ]:
# -------- Step 5: Evaluate CNN classification performance --------

from sklearn.metrics import classification_report, confusion_matrix

# ---- 1. Evaluate directly using Keras ----
print("===== Keras Evaluation =====")
train_loss, train_acc = model.evaluate(X_train_cnn, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val_cnn, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)

print(f"Train  -> Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
print(f"Val    -> Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f}")
print(f"Test   -> Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f}")

# ---- 2. Detailed metrics (Precision, Recall, F1) ----

# Predict labels
y_train_pred = model.predict(X_train_cnn).argmax(axis=1)
y_val_pred   = model.predict(X_val_cnn).argmax(axis=1)
y_test_pred  = model.predict(X_test_cnn).argmax(axis=1)

print("\n===== Classification Report: TRAIN =====")
print(classification_report(y_train, y_train_pred))

print("\n===== Classification Report: VALIDATION =====")
print(classification_report(y_val, y_val_pred))

print("\n===== Classification Report: TEST =====")
print(classification_report(y_test, y_test_pred))

# ---- 3. Optional Confusion Matrix ----
print("\n===== Test Confusion Matrix =====")
print(confusion_matrix(y_test, y_test_pred))


===== Keras Evaluation =====
Train  -> Loss: 0.1100 | Accuracy: 0.9626
Val    -> Loss: 57.6116 | Accuracy: 0.9013
Test   -> Loss: 0.6632 | Accuracy: 0.9056
218/218 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step

===== Classification Report: TRAIN =====
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       752
           1       0.98      0.94      0.96      1260
           2       0.94      1.00      0.97      2342
           3       0.99      0.94      0.96      1527
           4       0.99      0.94      0.96      1077

    accuracy                           0.96      6958
   macro avg       0.96      0.96      0.96      6958
weighted avg       0.96      0.96      0.96      6958


===== Classification Report: VALIDATION =====
              precision    recall  f1-score   support

           0       0.77      0.87      0.82       250
           1       0.91      0.86  

In [ ]:


# -------- Step 6: Save CNN & Embedding Models and Embedding CSVs (400 Features) --------

import os
import pandas as pd

# Make sure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- SAVE MODELS ----------------

# 1️⃣ Save trained CNN classifier(full trained CNN classifier model)
cnn_model_path = os.path.join(OUTPUT_DIR, "malware_cnn_model_374Features.h5")
model.save(cnn_model_path)
print("Saved CNN model to:", cnn_model_path)

# 2️⃣ Save embedding model
embedding_model_path = os.path.join(OUTPUT_DIR, "malware_embedding_model_374Features.h5")
embedding_model.save(embedding_model_path)
print("Saved embedding model to:", embedding_model_path)


# ---------------- SAVE EMBEDDING CSV FILES ----------------

# train embeddings
train_csv_path = os.path.join(OUTPUT_DIR, "train_embeddings_374Features.csv")
pd.DataFrame(train_embeddings).to_csv(train_csv_path, index=False)

# validation embeddings
val_csv_path = os.path.join(OUTPUT_DIR, "val_embeddings_374Features.csv")
pd.DataFrame(val_embeddings).to_csv(val_csv_path, index=False)

# test embeddings
test_csv_path = os.path.join(OUTPUT_DIR, "test_embeddings_374Features.csv")
pd.DataFrame(test_embeddings).to_csv(test_csv_path, index=False)

print("\nEmbedding CSV files saved:")
print(train_csv_path)
print(val_csv_path)
print(test_csv_path)


Saved CNN model to: /content/drive/MyDrive/malware_project/malware_cnn_model_374Features.h5
Saved embedding model to: /content/drive/MyDrive/malware_project/malware_embedding_model_374Features.h5

Embedding CSV files saved:
/content/drive/MyDrive/malware_project/train_embeddings_374Features.csv
/content/drive/MyDrive/malware_project/val_embeddings_374Features.csv
/content/drive/MyDrive/malware_project/test_embeddings_374Features.csv


In [1]:
#DEEP FEATURE EXTRACTION LATENCY MEASUREMENT******************************************
# =========================================================
# 0️⃣ Mount Google Drive
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

import time
import numpy as np
import pandas as pd
import tensorflow as tf
import os

# =========================================================
# 1️⃣ Set Base Path (CHANGE if your folder name is different)
# =========================================================
base_path = "/content/drive/MyDrive/malware_project"

# =========================================================
# 2️⃣ Load Saved Embedding Model
# =========================================================
embedding_model = tf.keras.models.load_model(
    os.path.join(base_path, "malware_embedding_model_374Features.h5")
)

print("✅ Model loaded successfully.")
print("Expected input shape:", embedding_model.input_shape)

# =========================================================
# 3️⃣ Load RAW Split Datasets
# =========================================================
train_data = pd.read_csv(os.path.join(base_path, "train_374Features.csv"))
val_data   = pd.read_csv(os.path.join(base_path, "val_374Features.csv"))
test_data  = pd.read_csv(os.path.join(base_path, "test_374Features.csv"))

print("Original train shape:", train_data.shape)

# =========================================================
# 4️⃣ Remove Label Column
# =========================================================
X_train = train_data.iloc[:, :-1].values
X_val   = val_data.iloc[:, :-1].values
X_test  = test_data.iloc[:, :-1].values

print("Features shape after label removal:", X_train.shape)

# =========================================================
# 5️⃣ Reshape for CNN Input (374 → 374x1)
# =========================================================
feature_count = X_train.shape[1]

X_train = X_train.reshape(-1, feature_count, 1)
X_val   = X_val.reshape(-1, feature_count, 1)
X_test  = X_test.reshape(-1, feature_count, 1)

print("Final CNN input shape:", X_train.shape)

# =========================================================
# 6️⃣ Measure Embedding Extraction Runtime
# =========================================================
start_time = time.time()

train_emb = embedding_model.predict(X_train, batch_size=32, verbose=0)
val_emb   = embedding_model.predict(X_val, batch_size=32, verbose=0)
test_emb  = embedding_model.predict(X_test, batch_size=32, verbose=0)

end_time = time.time()

# =========================================================
# 7️⃣ Runtime Statistics
# =========================================================
total_time = end_time - start_time
total_samples = len(X_train) + len(X_val) + len(X_test)

print("\n✅ Embedding Extraction Complete!")
print("Total Time:", round(total_time, 4), "seconds")
print("Total Samples:", total_samples)
print("Time per Sample:", round(total_time / total_samples, 8), "seconds")
print("Throughput:", round(total_samples / total_time, 2), "samples/second")


Mounted at /content/drive


✅ Model loaded successfully.
Expected input shape: (None, 374, 1)
Original train shape: (6958, 375)
Features shape after label removal: (6958, 374)
Final CNN input shape: (6958, 374, 1)

✅ Embedding Extraction Complete!
Total Time: 7.1461 seconds
Total Samples: 11598
Time per Sample: 0.00061615 seconds
Throughput: 1622.98 samples/second
